# Module 04.16 — Event Annotation Groups

> Part of the **Elasticsearch Snapshot Training** course.
> Run `00_setup.ipynb` first to start the Docker stack and load sample data.


In [ ]:
import sys, json, time
sys.path.insert(0, '..')
from helpers import *

es = get_client()

---
## 4.16 Event Annotation Groups (`event-annotation-group`)

Event annotations mark **specific time points** on Lens time series charts — as vertical
lines with optional labels and icons. They are useful for overlaying deployment events,
incidents, or business milestones on a metrics chart, so you can visually correlate a
spike in error rate with the deploy that caused it.

Annotations come in two flavours:
- **Manual** — a fixed timestamp with a label, stored entirely in the saved object
- **Query-based** — a KQL query run against an index; every matching document's timestamp
  becomes an annotation. Query-based annotations hold a reference to a Data View.

Both flavours are grouped inside an `event-annotation-group` object, which can be reused
across multiple Lens charts. The group itself references the Data View used for
query-based annotations (if any); manual-only groups have no external references.

From a restore standpoint, event annotation groups behave like any other Lens dependency:
they must be restored before the Lens charts that reference them, and they in turn depend
on their Data View being present first. The feature-state restore handles this ordering
automatically.

To create an annotation group in the Lens editor: open a time series chart, click
**Annotations** in the layer panel, add a layer, and save the group to the library so it
can be shared with other charts.

In [ ]:
heading('4.16 Event Annotation Groups')

annotations = find_saved_objects('event-annotation-group')
console.print(f'  Found {len(annotations)} event-annotation-group object(s)')
if annotations:
    pp(annotations[0], 'event-annotation-group saved object')
    info('Key fields:')
    info('  annotations[] — array of annotation definitions')
    info('    type: manual | query')
    info('    manual: fixed timestamp, label, line style/colour')
    info('    query: ES query that resolves to event timestamps at render time')
    info('  dataViewSpec  — embedded data view spec (or reference to existing data view)')
    info('references      — link to index-pattern if using a referenced data view')
else:
    info('No annotation groups yet — create a Lens chart and add an event annotation to see this type.')

In [ ]:
heading('4.16 Event Annotation Group — create via API')

ecom_dv = next((o for o in find_saved_objects('index-pattern') if 'ecommerce' in o['attributes'].get('title', '')), None)
if not ecom_dv:
    raise RuntimeError('eCommerce data view not found — re-run 00_setup')
ecom_dv_id = ecom_dv['id']

existing_ann = next((o for o in find_saved_objects('event-annotation-group') if o['attributes'].get('title') == 'Training — Key Events'), None)
if existing_ann:
    ann_id = existing_ann['id']
    info(f'Annotation group already exists: id={ann_id}')
else:
    ann_resp = kibana_post('/api/saved_objects/event-annotation-group', {
        'attributes': {
            'title': 'Training — Key Events',
            'description': 'Key events for snapshot training exercises',
            'ignoreGlobalFilters': False,
            'annotations': [{
                'id': 'ann-1',
                'type': 'manual',
                'key': {'type': 'point_in_time', 'timestamp': '2023-12-01T00:00:00.000Z'},
                'label': 'Training Start',
                'icon': 'triangle',
                'color': '#0077cc',
                'isHidden': False,
            }],
            'dataViewSpec': None,
        },
        'references': [{'id': ecom_dv_id, 'name': 'event-annotation-group-index-pattern', 'type': 'index-pattern'}],
    })
    ann_id = ann_resp['id']
    success(f'Event annotation group created: id={ann_id}')

obj = kibana_get(f'/api/saved_objects/event-annotation-group/{ann_id}')
pp(obj, 'event-annotation-group saved object')
info('Key fields:')
info('  annotations  — array of manual or query-based annotations')
info('    type=manual  → point-in-time marker with timestamp')
info('    type=query   → dynamic markers from an ES query against a data view')
info('  references   — data view used by query-based annotations')

In [ ]:
# Ensure annotation group exists before snapshotting (re-runnable)
if not any(o['id'] == ann_id for o in find_saved_objects('event-annotation-group')):
    warn('Annotation group missing — recreating')
    ecom_dv_id = next(o['id'] for o in find_saved_objects('index-pattern') if 'ecommerce' in o['attributes'].get('title', ''))
    ann_resp = kibana_post('/api/saved_objects/event-annotation-group', {
        'attributes': {
            'title': 'Training — Key Events', 'description': 'Key events for training',
            'ignoreGlobalFilters': False,
            'annotations': [{'id': 'ann-1', 'type': 'manual',
                'key': {'type': 'point_in_time', 'timestamp': '2023-12-01T00:00:00.000Z'},
                'label': 'Training Start', 'icon': 'triangle', 'color': '#0077cc', 'isHidden': False}],
            'dataViewSpec': None,
        },
        'references': [{'id': ecom_dv_id, 'name': 'event-annotation-group-index-pattern', 'type': 'index-pattern'}],
    })
    ann_id = ann_resp['id']

snap_delete_restore_cycle('snap-annotation-test', 'Event Annotation Group')

kibana_delete(f'/api/saved_objects/event-annotation-group/{ann_id}')
warn(f'Accidentally deleted annotation group {ann_id}')
assert not any(o['id'] == ann_id for o in find_saved_objects('event-annotation-group')), 'Should be gone'

restore_kibana_state(es, SNAP_REPO, 'snap-annotation-test')
time.sleep(3)

restored = find_saved_objects('event-annotation-group')
assert any(o['id'] == ann_id for o in restored), 'Annotation group should be restored'
success(f'Event annotation group {ann_id} successfully restored!')